# Video Cut AI — Colab inference server

Запускайте комірки зверху вниз. Сервер приймає відео через `POST /predict`, обробляє їх послідовно на GPU та повертає результат через `GET /result/<task_id>`.

Перед запуском додайте `NGROK_AUTHTOKEN` у **Colab → Secrets**. Не вставляйте token безпосередньо в код.

In [ ]:
!pip -q install ultralytics flask flask-cors pyngrok
!ffmpeg -version | head -n 1

In [ ]:
from pathlib import Path
import os

# Якщо модель лежить у Google Drive, спочатку розкоментуйте mount і змініть MODEL_PATH.
# from google.colab import drive
# drive.mount('/content/drive')
# MODEL_PATH = Path('/content/drive/MyDrive/path/to/best2.pt')

MODEL_PATH = Path('/content/best2.pt')
DEFAULT_FPS = 1.0
DEFAULT_IMAGE_SIZE = 320
DEFAULT_MIN_DURATION = 2.0
DEFAULT_SMOOTH_WINDOW = 3
QUEUE_MAX = 4
RESULT_TTL_SECONDS = 6 * 60 * 60
UPLOAD_DIR = Path('/content/video-cut-ai-uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    from google.colab import files
    print(f'Model not found at {MODEL_PATH}. Upload a .pt file now.')
    uploaded = files.upload()
    pt_files = [Path('/content') / name for name in uploaded if name.lower().endswith('.pt')]
    if not pt_files:
        raise FileNotFoundError('No .pt model was uploaded')
    MODEL_PATH = pt_files[0]

try:
    from google.colab import userdata
    NGROK_AUTHTOKEN = (userdata.get('NGROK_AUTHTOKEN') or '').strip()
except Exception:
    NGROK_AUTHTOKEN = os.environ.get('NGROK_AUTHTOKEN', '').strip()

if not NGROK_AUTHTOKEN:
    raise RuntimeError('Add NGROK_AUTHTOKEN in Colab Secrets and allow notebook access')

print('Model:', MODEL_PATH)
print('Upload directory:', UPLOAD_DIR)

In [ ]:
import math
import queue
import subprocess
import tempfile
import threading
import time
import uuid
from collections import Counter

import torch
from flask import Flask, jsonify, request
from flask_cors import CORS
from ultralytics import YOLO
from werkzeug.utils import secure_filename

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Loading YOLO model on', DEVICE)
model = YOLO(str(MODEL_PATH))
print('Model loaded. Classes:', model.names)

old_queue = globals().get('task_queue')
if old_queue is not None:
    try:
        old_queue.put_nowait(None)
    except Exception:
        pass

task_queue = queue.Queue(maxsize=QUEUE_MAX)
results_store = {}
results_lock = threading.Lock()

def set_task(task_id, **updates):
    with results_lock:
        task = results_store.setdefault(task_id, {'created_at': time.time()})
        task.update(updates)

def get_task(task_id):
    with results_lock:
        task = results_store.get(task_id)
        return dict(task) if task else None

def cleanup_old_results():
    cutoff = time.time() - RESULT_TTL_SECONDS
    with results_lock:
        expired = [task_id for task_id, data in results_store.items() if data.get('created_at', 0) < cutoff]
        for task_id in expired:
            results_store.pop(task_id, None)

def normalize_label(label):
    value = str(label or '').strip().lower().replace('-', '_').replace(' ', '_')
    thermal = 'thermal' in value or 'infrared' in value or value.startswith('ir_')
    if thermal and 'analog' in value:
        return 'thermal_analog'
    if thermal and 'digital' in value:
        return 'thermal_digital'
    if 'analog' in value:
        return 'analog'
    if 'digital' in value:
        return 'digital'
    return value

def clamp_number(value, fallback, minimum, maximum):
    try:
        number = float(value)
    except (TypeError, ValueError):
        number = fallback
    return max(minimum, min(maximum, number))

def extract_frames(video_path, out_dir, fps, image_size):
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = out_dir / '%06d.jpg'
    command = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-nostdin',
        '-i', str(video_path),
        '-vf', f'fps={fps},scale={image_size}:-2',
        '-q:v', '2', str(pattern)
    ]
    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.returncode != 0:
        detail = (completed.stderr or 'unknown ffmpeg error')[-1200:]
        raise RuntimeError(f'FFmpeg failed: {detail}')

def majority_smooth(labels, window):
    if len(labels) < 3 or window <= 1:
        return list(labels)
    window = max(1, int(window))
    if window % 2 == 0:
        window += 1
    radius = window // 2
    smoothed = []
    for index, current in enumerate(labels):
        nearby = labels[max(0, index - radius):min(len(labels), index + radius + 1)]
        counts = Counter(nearby)
        winner = max(counts, key=lambda label: (counts[label], label == current))
        smoothed.append(winner)
    return smoothed

def build_segments(labels, fps, min_duration):
    if not labels:
        return {}, 0
    minimum_frames = max(1, math.ceil(min_duration * fps))
    segments = {}
    dropped_runs = 0
    start = 0
    current = labels[0]

    def add_run(label, first_frame, end_frame):
        nonlocal dropped_runs
        frame_count = end_frame - first_frame
        if frame_count < minimum_frames:
            dropped_runs += 1
            return
        segments.setdefault(label, []).append({
            'id': f'{label}_{first_frame}_{end_frame}',
            'start': round(first_frame / fps, 3),
            'end': round(end_frame / fps, 3)
        })

    for index in range(1, len(labels)):
        if labels[index] != current:
            add_run(current, start, index)
            start = index
            current = labels[index]
    add_run(current, start, len(labels))
    return segments, dropped_runs

def process_video(task):
    task_id = task['id']
    video_path = Path(task['video'])
    set_task(task_id, status='processing', started_at=time.time())
    if not video_path.exists():
        raise FileNotFoundError(f'Queued video disappeared: {video_path}')

    with tempfile.TemporaryDirectory(prefix=f'{task_id}_frames_') as tmp_dir:
        frames_dir = Path(tmp_dir) / 'frames'
        extract_frames(video_path, frames_dir, task['fps'], task['image_size'])
        frames = sorted(frames_dir.glob('*.jpg'))
        if not frames:
            raise RuntimeError('FFmpeg completed but extracted 0 frames')
        set_task(task_id, frame_count=len(frames))

        labels = []
        confidences = []
        batch_size = 32
        for offset in range(0, len(frames), batch_size):
            batch = [str(path) for path in frames[offset:offset + batch_size]]
            predictions = model.predict(batch, verbose=False, device=DEVICE)
            for prediction in predictions:
                if prediction.probs is None:
                    raise RuntimeError('Loaded YOLO model is not a classification model: prediction.probs is empty')
                class_index = int(prediction.probs.top1)
                labels.append(normalize_label(model.names[class_index]))
                confidences.append(float(prediction.probs.top1conf))
            set_task(task_id, processed_frames=min(offset + batch_size, len(frames)))

        smoothed_labels = majority_smooth(labels, task['smooth_window'])
        segments, dropped_runs = build_segments(smoothed_labels, task['fps'], task['min_duration'])
        set_task(
            task_id,
            status='done',
            result=segments,
            label_counts=dict(Counter(smoothed_labels)),
            average_confidence=round(sum(confidences) / len(confidences), 4) if confidences else None,
            dropped_runs=dropped_runs,
            finished_at=time.time()
        )

def gpu_worker():
    while True:
        task = task_queue.get()
        if task is None:
            task_queue.task_done()
            break
        try:
            process_video(task)
        except Exception as error:
            set_task(task['id'], status='error', error=str(error), finished_at=time.time())
        finally:
            try:
                Path(task['video']).unlink(missing_ok=True)
            except Exception:
                pass
            task_queue.task_done()

worker_thread = threading.Thread(target=gpu_worker, name='video-cut-gpu-worker', daemon=True)
worker_thread.start()

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 3 * 1024 * 1024 * 1024
CORS(app)

@app.get('/health')
def health():
    cleanup_old_results()
    with results_lock:
        processing = sum(1 for item in results_store.values() if item.get('status') == 'processing')
    return jsonify({
        'ok': True,
        'queue_size': task_queue.qsize(),
        'queue_max': QUEUE_MAX,
        'processing': processing,
        'device': str(DEVICE),
        'model': MODEL_PATH.name,
        'classes': model.names
    })

@app.post('/predict')
def predict():
    cleanup_old_results()
    if 'video' not in request.files:
        return jsonify({'ok': False, 'error': 'missing_video'}), 400
    if task_queue.full():
        return jsonify({'ok': False, 'error': 'server_busy'}), 429

    uploaded_file = request.files['video']
    original_name = secure_filename(uploaded_file.filename or 'video.mp4') or 'video.mp4'
    suffix = Path(original_name).suffix or '.mp4'
    fps = clamp_number(request.form.get('fps'), DEFAULT_FPS, 0.1, 10.0)
    image_size = int(clamp_number(request.form.get('size'), DEFAULT_IMAGE_SIZE, 160, 1280))
    min_duration = clamp_number(request.form.get('min_duration'), DEFAULT_MIN_DURATION, 0.0, 600.0)
    smooth_window = int(clamp_number(request.form.get('smooth_window'), DEFAULT_SMOOTH_WINDOW, 1, 15))
    task_id = f'task_{uuid.uuid4().hex[:16]}'

    file_descriptor, stored_name = tempfile.mkstemp(prefix=f'{task_id}_', suffix=suffix, dir=UPLOAD_DIR)
    os.close(file_descriptor)
    video_path = Path(stored_name)
    try:
        uploaded_file.save(video_path)
        set_task(
            task_id,
            status='queued',
            file_name=original_name,
            fps=fps,
            image_size=image_size,
            min_duration=min_duration,
            smooth_window=smooth_window
        )
        task_queue.put_nowait({
            'id': task_id,
            'video': str(video_path),
            'fps': fps,
            'image_size': image_size,
            'min_duration': min_duration,
            'smooth_window': smooth_window
        })
    except queue.Full:
        video_path.unlink(missing_ok=True)
        with results_lock:
            results_store.pop(task_id, None)
        return jsonify({'ok': False, 'error': 'server_busy'}), 429
    except Exception:
        video_path.unlink(missing_ok=True)
        with results_lock:
            results_store.pop(task_id, None)
        raise

    return jsonify({'ok': True, 'task_id': task_id, 'status': 'queued'}), 202

@app.get('/result/<task_id>')
def result(task_id):
    task = get_task(task_id)
    if task is None:
        return jsonify({'ok': False, 'error': 'not_found'}), 404
    task.pop('created_at', None)
    return jsonify({'ok': True, **task})

print('API and GPU worker are ready')

In [ ]:
import requests
from pyngrok import ngrok
from werkzeug.serving import make_server

class FlaskServerThread(threading.Thread):
    def __init__(self, flask_app, host='0.0.0.0', port=5000):
        super().__init__(daemon=True)
        self.server = make_server(host, port, flask_app, threaded=True)
        self.context = flask_app.app_context()
        self.context.push()

    def run(self):
        self.server.serve_forever()

    def shutdown(self):
        self.server.shutdown()

old_server = globals().get('flask_server')
if old_server is not None:
    try:
        old_server.shutdown()
    except Exception:
        pass

try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTHTOKEN)
flask_server = FlaskServerThread(app)
flask_server.start()
public_tunnel = ngrok.connect(5000, bind_tls=True)
PUBLIC_URL = public_tunnel.public_url.rstrip('/')

health_response = requests.get('http://127.0.0.1:5000/health', timeout=10).json()
print('Health:', health_response)
print('\nPaste this URL into Video Cut App:')
print(PUBLIC_URL)

## Використання

1. Скопіюйте надрукований `PUBLIC_URL`.
2. Вставте URL у **Адмінка → AI сегменти → Colab URL**.
3. Натисніть **Ping**, потім **AI Colab**.
4. Не закривайте Colab runtime, поки черга працює.

Якщо модель поверне нуль сегментів, `/result` покаже `frame_count`, `label_counts`, `average_confidence` та `dropped_runs`, тому причина буде видима.